# Blokk titkosítók

## Feistel hálózatok

![Feistel hálózatok](./feistel_network.png)

## Confusion és Diffusion

Claude Shannon 1945:
- confusion: a ciphertext minden bitje a kulcs néhány részétől függ
- diffusion: ha megváltoztatjuk a plaintext egy bitjét, akkor a ciphertext bitjeinek fele meg kell hogy változzon. Ugyanígy, ha a ciphertext egy bitjét megváltoztatjuk, akkor a plaintext bitjeinek fele meg kell hogy változzon.

## bytestring Pythonban

Bájtsorozatok, ASCII karaktereken. [Dokumentáció](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview)

![ASCII tábla](./ascii_table.png)

In [1]:
b"crypto\xff\x5f"

b'crypto\xff_'

In [2]:
b"crpyto".hex()

'63727079746f'

Ha a bytestring-en végigiterálunk, a decimális reprezentációját kapjuk az egyes karaktereknek:

In [3]:
for i in b"crypto\xff":
    print(i)

99
114
121
112
116
111
255


Az indexelés is a decimális értékét írja ki a bytenak:

In [4]:
v = b"crypto\xff"
v[-1], v[0]

(255, 99)

**Feladat**: XOR-ozzunk össze két sztringet!

Hexadecimális számok:

In [5]:
for i in range(20):
    print(f'{i:>2}: {i:08b} {i:02x}')

 0: 00000000 00
 1: 00000001 01
 2: 00000010 02
 3: 00000011 03
 4: 00000100 04
 5: 00000101 05
 6: 00000110 06
 7: 00000111 07
 8: 00001000 08
 9: 00001001 09
10: 00001010 0a
11: 00001011 0b
12: 00001100 0c
13: 00001101 0d
14: 00001110 0e
15: 00001111 0f
16: 00010000 10
17: 00010001 11
18: 00010010 12
19: 00010011 13


Integert hexadecimális számmá konvertálni a `hex()` függvénnyel lehet:

In [6]:
hex(137), bin(137)

('0x89', '0b10001001')

In [7]:
int('0x89', base=16), int('0b10001001', base=2)

(137, 137)

In [8]:
int(137).to_bytes(1)

b'\x89'

In [9]:
bytes.fromhex("fe89")

b'\xfe\x89'

In [10]:
def xor_bytestrings(a, b):
    return b"".join([(int(x) ^^ int(y)).to_bytes(1) for (x,y) in zip(a, b)])

In [11]:
xor_bytestrings(b"\xff", b"\x00")

b'\xff'

## PKCS #5 és PKCS #7 padding

PKCS: Public Key Cryptography Standards

Cél: a blokkméret többszöröse legyen a plaintext hossza

**Példa**: Legyen a blokkméret 8 bájt, de az üzenet 12 bájt. Ekkor a padding:

`DD DD DD DD DD DD DD DD | DD DD DD DD 04 04 04 04 |`

Ha az üzenet blokkméret hosszú, akkor egy új blokkot adunk hozzá

In [12]:
def padding(block_size: int, msg):
    p = block_size - (len(msg) % block_size)
    return msg + bytes.fromhex(f"{p:02x}") * p

In [13]:
size = 8
padding(size, b"cryptohraphy")
padding(12, b"cryptohraphy")
#padding(size, b"")

b'cryptohraphy\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c'

Egyéb padding módszerek:
- bit padding: egy darab `1` utána annyi `0`, amennyi kell
- ISO/IEC 7816-4: `80` után annyi `0`, amennyi kell
- Zero padding: annyi `0`, amennyi kell

## Data Encryption Standard

1977, [NIST standard (pdf)](https://csrc.nist.gov/CSRC/media/Publications/fips/46/3/archive/1999-10-25/documents/fips46-3.pdf)

Tulajdonságok:
- 64 (56) bit hosszú kulcs
- 64 bit hosszú blokk
- Feistel-hálózat 16 körrel

![DES](./des.png)

In [14]:
from operator import xor
import secrets
from typing import List, Optional

from permutations import *

In [15]:
class BadPaddingError(Exception):
    pass

In [16]:
def xor_blocks(a: List[int], b: List[int]) -> List[int]:
    return [xor(i, j) for i, j in zip(a, b)]


def check_padding(s: bytes) -> bool:
    p = s[-1]
    return all(map(lambda x: x == p, s[-p:]))


def xor_strings(a, b):
    m = b""
    for i, j in zip(a, b):
        m += bytes.fromhex(f'{xor(i, j):02x}')
    return m

In [17]:
check_padding(b'cryptohraphy\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c\x0c')

True

In [18]:
class DES:
    def __init__(self, key: bytes, block_size: int) -> None:
        if len(key) != 8:
            raise ValueError(f'The key must be 8 bytes long')
        self.key = key
        self.block_size = block_size
        self.round_keys = self._key_schedule()
    
    def encrypt(self, plaintext: bytes, apply_pad: bool) -> bytes:
        pass
    
    def decrypt(self, ciphertext: bytes, apply_pad: bool) -> bytes:
        pass
        
    def _encryption(self, block: bytes, encrypt: bool) -> bytes:
        block = int.from_bytes(block, byteorder='big')
        block = list(map(int, f'{block:064b}'))
        block = [block[i] for i in IP]
        
        if not encrypt:
            keys = self.round_keys[::-1]
        else:
            keys = self.round_keys
        
        left, right = block[:32], block[32:]
        for i in range(16):
            round_key = keys[i]
            out = self._cipher(right, round_key)
            out = xor_strings(left, out)
            left, right = right, out
        
        block = right + left
        block = [block[i] for i in IP_inv]
        block = '0b' + ''.join([str(i) for i in block])
        block = int(block, base=2).to_bytes(8, 'big')
        return block
        
    def _cipher(self, r: List[int], k: List[int]) -> List[int]:
        subst = [S1, S2, S3, S4, S5, S6, S7, S8]
        r = [r[i] for i in E]
        rk = xor_blocks(r, k)
        res, j = [], 0
        for i in range(0, 48 - 6 + 1, 6):
            block = rk[i:i+6]
            row, col = int(f'0b{block[0]}{block[-1]}', base=2), int(f'0b{"".join(str(b) for b in block[1:-1])}', base=2)
            res += [int(b) for b in bin(subst[j][row][col])[2:].zfill(4)]
            j += 1
        res = [res[i] for i in P]
        return res
    
    def _key_schedule(self) -> List[List[int]]:
        iterations = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]
        res = []
        key = int.from_bytes(self.key, byteorder='big')
        key = list(map(int, bin(key)[2:].zfill(64)))
        round_key = [key[i] for i in PC1]
        c, d = round_key[:28], round_key[28:]
        for n in range(16):
            left_shifts = iterations[n]
            c = c[left_shifts:] + c[:left_shifts]
            d = d[left_shifts:] + d[:left_shifts]
            round_key = c + d
            res.append([round_key[i] for i in PC2])
        return res

## Működési módok

### Electronic Code Book (ECB)

![ECB mód](./mode_ecb.png)

A titkosított szöveg: $c = F(k, m_1)\,||\,F(k, m_2)\,||\,\cdots\,||\,F(k, m_\ell))$

Visszafejtés: $p = F^{-1}(k, c_1)\,||\,F^{-1}(k, c_2)\,||\,\cdots\,||\,F^{-1}(k, c_{\ell})$

### Cipher Block Chaining (CBC)

![CBC mód](./mode_cbc.png)

A titkosított szöveg több részből áll:
- $c_0 = IV$
- $c_i = F(k, c_{i-1} \oplus m_{i})$
- és $c = c_0\,||\,c_1 ||\,\cdots\,||c_{\ell}$

Visszafejtés: $m_i = F^{-1}(k,c_i) \oplus c_{i-1}$ minden $1 \leq i \leq \ell$ blokkra

**Feladat**: Titkosítsuk AES titkosítással ECB és CBC módban az ELTE címerét!

**Megoldás**:
1. ECB mód:
```bash
head -n 4 elte_cimer.ppm > header.txt
tail -n +5 elte_cimer.ppm > body.bin
openssl enc -aes-128-ecb -nosalt -pass pass:"VeryLongPassword" -in body.bin -out body.ecb.bin
cat header.txt body.ecb.bin > elte_cimer_ecb.ppm
```
2. CBC mód (az IV random generálódik):
```bash
head -n 4 elte_cimer.ppm > header.txt
tail -n +5 elte_cimer.ppm > body.bin
openssl enc -aes-128-cbc -nosalt -pass pass:"VeryLongPassword" -in body.bin -out body.cbc.bin
cat header.txt body.cbc.bin > elte_cimer_cbc.ppm
```

In [19]:
class DES_ECB(DES):
    def __init__(self, key: bytes, block_size: int = 8) -> None:
        super().__init__(key, block_size)
        
    def encrypt(self, plaintext: bytes, apply_pad: bool = True) -> bytes:
        if apply_pad and not check_padding(plaintext):
            raise BadPaddingError()
        
        blocks = [plaintext[i:i+self.block_size] for i in range(0, len(plaintext), self.block_size)]
        encrypted_blocks = list(map(lambda b: self._encryption(b, encrypt=True), blocks))
        
        return b"".join(encrypted_blocks)
    
    def decrypt(self, ciphertext: bytes, apply_pad: bool = True) -> bytes:        
        encrypted_blocks = [ciphertext[i:i+self.block_size] for i in range(0, len(ciphertext), self.block_size)]
        decrypted_blocks = list(map(lambda b: self._encryption(b, encrypt=False), encrypted_blocks))
        
        decrypted = b"".join(decrypted_blocks)
        
        if apply_pad: 
            if check_padding(decrypted):
                return decrypted[:-decrypted[-1]]
            else:
                raise BadPaddingError()
        
        return decrypted
        

In [20]:
plaintext = b'cryptography'
des_ecb = DES_ECB(b'passwd01')
ciphertext = des_ecb.encrypt(padding(8, plaintext))
des_ecb.decrypt(ciphertext) == plaintext

True

In [21]:
class DES_CBC(DES):
    def __init__(self, key: bytes, block_size: int = 8, iv: Optional[bytes] = None) -> None:
        super().__init__(key, block_size)
        if iv is None:
            self.iv = secrets.token_bytes(self.block_size)
        else:
            self.iv = iv
        
    def encrypt(self, plaintext: bytes, apply_pad: bool = True) -> bytes:
        if apply_pad and not check_padding(plaintext):
            raise BadPaddingError()
        
        ct = self.iv
        
        for block in range(0, len(plaintext), self.block_size):
            pt_block = plaintext[block:block+self.block_size]
            inp = xor_strings(pt_block, ct[-self.block_size:])
            ct += self._encryption(inp, True)
        
        return ct
    
    def decrypt(self, ciphertext: bytes, apply_pad: bool = True) -> bytes:
        pt = b''
        for block in range(self.block_size, len(ciphertext), self.block_size):
            ct_block = ciphertext[block:block+self.block_size]
            res = self._encryption(ct_block, False)
            pt += xor_strings(res, ciphertext[block-self.block_size:block])
            
        if apply_pad: 
            if check_padding(pt):
                return pt[:-pt[-1]]
            else:
                raise BadPaddingError()
        
        return pt

In [22]:
plaintext = b'cryptography'

des_cbc = DES_CBC(b'passwd01')
ciphertext = des_cbc.encrypt(padding(8, plaintext))
des_cbc.decrypt(ciphertext) == plaintext

True

In [23]:
des_cbc = DES_CBC(b'passwd01', iv=b'hellothe')
ciphertext = des_cbc.encrypt(padding(8, plaintext))
des_cbc.decrypt(ciphertext) == plaintext

True

### Gyenge és félgyenge kulcsok

Legyenek $w_1,w_2$ kulcsok és $m$ egy üzenet. 
- ha $\text{DES}(w_1, \text{DES}(w_1, m)) = m,$ akkor a $w_1$ kulcs gyenge. 
- ha $\text{DES}(w_1, \text{DES}(w_2, m)) = m,$ akkor a $w_1, w_2$ kulcspár gyenge kulcspár.

Gyenge kulcsok:
- `0101010101010101`
- `fefefefefefefefe`
- `1f1f1f1f0e0e0e0e`
- `e0e0e0e0f1f1f1f1`

Félgyenge kulcspárok:
- `01fe01fe01fe01fe` és `fe01fe01fe01fe01`
- `1fe01fe00ef10ef1` és `e01fe01ff10ef10e`
- `01e001e001f001f1` és `e001e001f101f101`
- `1ffe1ffe0efe0efe` és `fe1ffe1ffe0efe0e`
- `011f011f010e010e` és `1f011f010e010e01`
- `e0fee0fef1fef1fe` és `fee0fee0fef1fef1`

## Padding-oracle attack CBC-módra

**Feladat**: Törjük fel a CBC-féle működési módot!

Tudjuk, hogy milyen hosszú egy blokk (ez a standard-okban benne van)

A titkosított szöveg több részből áll:
- $c_0 = IV$
- $c_i = F(k, c_{i-1} \oplus m_{i})$
- és $c = c_0\,||\,c_1 ||\,\cdots\,||c_{\ell}$

Visszafejtés: $m_i = F^{-1}(k,c_i) \oplus c_{i-1}$ minden $1 \leq i \leq \ell$ blokkra

- padding ellenőrzése visszafejtéskor: az utolsó $n$ bájt az $n$ értékét tartalmazza
- ötlet: bizonyos bájtok megváltoztatása a ciphertext-ben előrejelezhető változást okoz a visszafejtéskor


In [24]:
plaintext = b"cryptography"

pt = padding(8, plaintext)
pt

b'cryptography\x04\x04\x04\x04'

In [25]:
des = DES_CBC(b"mysecret", iv=b'hellothe')
ct = des.encrypt(pt, False)
ct, len(ct)

(b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e', 24)

**Példa**: Legyen a plaintext a fenti szöveg, a ciphertext ekkor $IV||c_1||c_2$.

A törés lépései:
1. Keressük meg a padding méretét (azaz a plaintext hosszát):
    - Módosítsuk az első bájtját $c_1$-nek és fejtsük vissza a módosított ciphertext-et
        - Ekkor $m_2' = F^{-1}(k, c_2) \oplus c_1'$, azaz $m_2$ első bájtja fog csak megváltozni
    - Ha nem kapunk hibát, akkor módosítsuk a következőt
    - Ahol először kapunk hibát, ott a padding rossz
        - Ebből tudjuk a padding (és plaintext) hosszát

In [26]:
for i in range(8):
    print(ct)
    ct_ = ct[:8+i] + b'\x00' + ct[8+i+1:]
    print(ct_)
    try:
        des.decrypt(ct_)
    except BadPaddingError:
        print(i)
        b = i
        break
    print("------")

b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
b'hellothe\x00\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
------
b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
b'hellothe\xa6\x00\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
------
b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
b'hellothe\xa6\xc3\x00Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
------
b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
b'hellothe\xa6\xc3\xc9\x00j\xde\xb4D)\xf7\xbf_Wse\x1e'
------
b'hellothe\xa6\xc3\xc9Ij\xde\xb4D)\xf7\xbf_Wse\x1e'
b'hellothe\xa6\xc3\xc9I\x00\xde\xb4D)\xf7\xbf_Wse\x1e'
4


In [27]:
pre = b'\x00' * (8 - (b + 1))

for i in range(256):
    left = pre + int(i).to_bytes(1, 'big') + int(b+1).to_bytes(1, 'big') * b
    right = pre + b'\x00' + int(b).to_bytes(1, 'big') * b
    delta_i = xor_strings(left, right)
    c1 = xor_strings(ct[8:16], delta_i)
    ct_ = ct[:8] + c1 + ct[16:]
    try:
        des.decrypt(ct_)
        print(xor_strings(int(b+1).to_bytes(1, 'big'), int(i).to_bytes(1, 'big')))
        break
    except BadPaddingError:
        continue

b'y'


2. Az üzenet utolsó bájtjának kiderítése
    - Tudjuk a padding hosszát, $b$-t. Jelöljük a plaintext utolsó bájtját $B$-vel
    - Tudjuk, hogy $m_2$ $0xB\,0xb \cdots 0xb$ végződésű ($0xb$ pontosan $b$-szer ismétlődik)
        - A példánkban: `\xB\x04\x04\x04\x04`
    - Legyen $\Delta_i = 0x00 \cdots 0x00\;0xi\;\underbrace{0x(b+1) \cdots 0x(b+1)}_\text{b-szer} \oplus 0x00 \cdots 0x00\;0x00\;\underbrace{0xb \cdots 0xb}_\text{b-szer}$ minden $0 \leq i < 2^8$. Az utolsó $b+1$ bájt az $i$-t és $(b+1) \oplus b$ értékét tartalmazza hexadecimális formában
        - Például: $\Delta_{137} = $ `\x00\x00\x00\x89\x05\x05\x05\x05 ^ \x00\x00\x00\x00\x04\x04\x04\x04 = \x00\x00\x00\x89\x01\x01\x01\x01`
    - Fejtsük vissza az $IV|| c_1 \oplus \Delta_i || c_2$ ciphertext-et
        - Az utolsó $b+1$ bájt $0x(B \oplus i)\;\underbrace{0x(b+1) \cdots 0x(b+1)}_\text{b-szer}$ alakú
        - A visszafejtés rossz lesz, amíg $0x(B \oplus i) = 0x(b+1)$ nem teljesül
        - Az utolsó bájt értéke $B = 0x(b+1) \oplus i$

## Meet-in-the-Middle attack

**Kérdés**: Adottak $k_1, k_2$ kulcsok valamilyen szimmetrikus titkosításhoz. Ad-e extra biztonságot, ha *kétszer* titkosítunk, azaz $c = \texttt{Enc}(k_2, \texttt{Enc}(k_1, \text{plaintext}))$ a következő titkosítóknál:
- eltolásos titkosítás
- affin titkosítás: $(a, b) \in \mathbb{Z}_{26}^* \times \mathbb{Z}_{26}^*$
    - $\mathtt{Enc}: c \equiv ap + b \pmod{26}$, 
    - $\mathtt{Dec}: p \equiv a^{-1}(c - b) \pmod{26}$
    - DES

A Meet-in-the-Middle attack módszere:

\begin{align*}
    c &= \mathtt{Enc}(k_2, \mathtt{Enc}(k_1, \text{plaintext})) \\
    \mathtt{Dec}(k_2, c) &= \mathtt{Dec}(k_2, \mathtt{Enc}(k_2, \mathtt{Enc}[k_1, \text{plaintext}])) \\
    \mathtt{Dec}(k_2, c) &= \mathtt{Enc}(k_1, \text{plaintext})
\end{align*}

Tegyük fel, hogy Eve ismeri az $m$ üzenetet és az ebből kapott $c$ titkosított üzenetet (ismert nyíltszövegű támadás). Cél: $k_1, k_2$ megtalálása.

A töréshez $\mathcal{O}\left( (n + \ell)\cdot 2^n \right)$ memóriára és $\mathcal{O}(n \cdot 2^n)$ időre van szükség.

Lépések:
1. Minden $k_1$ kulcsra számoljuk ki $z = \mathtt{Enc}(k_1, m)$ értékét és tároljuk el a $(z, k_1)$ párt egy $L$ listába.
2. Minden $k_2$ kulcsra számoljuk ki $z = \mathtt{Dec}(k_2, c)$ értékét és tároljuk el a $(z, k_2)$ párt egy $L'$ listába.
3. A $(z_1, k_1) \in L$ és $(z_2, k_2) \in L'$ elemek egyeznek, ha $z_1 = z_2$. Minden ilyen párnál adjuk $(k_1, k_2)$ párt egy $S$ halmazhoz.
    - Ez az $S$ halmaz csak azokat a $(k_1, k_2)$ párokat tartalmazza, amikre $\mathtt{Enc}(k_1, m) = \mathtt{Dec}(k_2, c)$
    - Ebben az $S$ halmazban benne lesz az ismeretlen kulcspár
4. Ha több $(m,c)$ párra meg tudjuk ezt csinálni, $S$ mérete jelentősen csökkenthető, ha vesszük a metszetüket.

In [43]:
import random
import secrets

des1 = DES_ECB(b'animator')
des2 = DES_ECB(b'reinvent')

plaintext = b'simplest'
ciphertext = des2.encrypt(des1.encrypt(plaintext, False), False)
ciphertext

b'o\x89R\x19\x90ih\x7f'

In [44]:
words = []
with open('/usr/share/dict/words') as infile:
    for word in infile:
        word = word.strip()
        if len(word) == 8:
            words.append(word)
words[:10]

["Aachen's",
 "Abbott's",
 'Aberdeen',
 "Abrams's",
 "Acadia's",
 'Acapulco',
 "Achebe's",
 'Achernar',
 'Achilles',
 "Acosta's"]

In [45]:
potential_pwds = random.sample(words, k=10) + ['animator', 'reinvent']
random.shuffle(potential_pwds)
potential_pwds

['implores',
 'stepdads',
 'flashier',
 "Haynes's",
 'imbedded',
 'rosebush',
 'immodest',
 'Islamist',
 'animator',
 "assize's",
 'reinvent',
 'solenoid']

In [47]:
for pk1 in potential_pwds:
    for pk2 in potential_pwds:
        
        d1 = DES_ECB(pk1.encode())
        d2 = DES_ECB(pk2.encode())
        
        ct = d2.encrypt(d1.encrypt(plaintext, False), False)
        
        if ct == ciphertext:
            print(f'Keys: {pk1}, {pk2}')
            break
            
z1_dict = {}
z2_dict = {}

for k1 in potential_pwds:
    k1 = k1.encode()
    d = DES_ECB(k1)
    
    z1 = d.encrypt(plaintext, False)
    z2 = d.decrypt(ciphertext, False)
    z1_dict[z1] = k1
    z2_dict[z2] = k1

for z1, k1 in z1_dict.items():
    for z2, k2 in z2_dict.items():
        if z1 == z2:
            print(k1, k2)
    

Keys: animator, reinvent


b'animator' b'reinvent'


És ha *háromszor* titkosítunk? Két opciónk van:
1. Válasszunk $k_1, k_2, k_3$ kulcsokat és legyen $c = \mathtt{Enc}(k_3, \mathtt{Dec}(k_2, \mathtt{Enc}(k_1, x)))$
    - A kulcs hossza $3n$
    - Meet-in-the-Middle támadást lehet alkalmazni ($\mathcal{O}(2^{2n})$ idő kell hozzá)
2. Válasszunk $k_1, k_2$ kulcsokat és legyen $c = \mathtt{Enc}(k_1, \mathtt{Dec}(k_2, \mathtt{Enc}(k_1, x)))$
    - A kulcs hossza $2n$
    - Nem ismert $\mathcal{O}(2^{2n})$-nél jobb támadás

3DES: 1999-ben standardizált, de lassú, a 2. verzió túl kis kulcsot használ